# Notebook 01: Preprocessing and Reproducibility

Create the clean project foundation: environment records, raw dataset audit, separate product/mask counts, MD5 and perceptual hashes, aspect-ratio-preserving 384x384 letterbox data, mask verification, and protocol notes.

## Path Setup

This cell is local/Colab adaptable. Change only `LAB_ROOT` if automatic detection does not find the parent folder.

In [1]:
from pathlib import Path
import sys

CURRENT_DIR = Path.cwd().resolve()

# Edit this one variable when moving between local execution and Colab.
LAB_ROOT = Path("PUT_PARENT_FOLDER_HERE")
if str(LAB_ROOT) == "PUT_PARENT_FOLDER_HERE":
    for candidate in [CURRENT_DIR, *CURRENT_DIR.parents]:
        if (candidate / "Phase 1").exists():
            LAB_ROOT = candidate
            break
    else:
        raise FileNotFoundError("Set LAB_ROOT to the parent folder containing 'Phase 1' and 'Project_A_LOCO_AD'.")

RAW_DATA_ROOT = LAB_ROOT / "Phase 1"
PROJECT_ROOT = LAB_ROOT / "Project_A_LOCO_AD"

NOTEBOOK_ROOT = PROJECT_ROOT / "01_notebooks"
AUDIT_ROOT = PROJECT_ROOT / "02_audit_reproducibility"
CLEAN_DATA_ROOT = PROJECT_ROOT / "03_cleaned_data"
PROBE_ROOT = PROJECT_ROOT / "04_probe_results"
BASELINE_ROOT = PROJECT_ROOT / "05_baselines"
METHOD_ROOT = PROJECT_ROOT / "06_method_results"
PAPER_ROOT = PROJECT_ROOT / "07_paper_draft"
EXPORT_ROOT = PROJECT_ROOT / "08_exports"

for folder in [NOTEBOOK_ROOT, AUDIT_ROOT, CLEAN_DATA_ROOT, PROBE_ROOT, BASELINE_ROOT, METHOD_ROOT, PAPER_ROOT, EXPORT_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)

if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

CATEGORIES = ["breakfast_box", "juice_bottle", "pushpins", "screw_bag", "splicing_connectors"]
for cat in CATEGORIES:
    print(cat, "OK" if (RAW_DATA_ROOT / cat).exists() else "MISSING")

breakfast_box OK
juice_bottle OK
pushpins OK
screw_bag OK
splicing_connectors OK


## Required Outputs

- `02_audit_reproducibility/environment_versions.csv`
- `02_audit_reproducibility/requirements_freeze.txt`
- `02_audit_reproducibility/dataset_audit_full.csv`
- `02_audit_reproducibility/dataset_product_images_only.csv`
- `02_audit_reproducibility/dataset_masks_only.csv`
- `02_audit_reproducibility/product_image_count_summary.csv`
- `02_audit_reproducibility/mask_count_summary.csv`
- `02_audit_reproducibility/resize_letterbox_metadata.csv`
- `02_audit_reproducibility/cleaned_letterbox_verify.csv`
- `02_audit_reproducibility/image_hashes.csv`
- `02_audit_reproducibility/config_preprocessing.json`
- `02_audit_reproducibility/preprocessing_protocol.txt`
- `03_cleaned_data/loco_cleaned_letterbox_384/`

## Acceptance Checks

- All five categories show OK.
- Product images = 3651, masks = 1246, total PNG = 4897, corrupted = 0.
- Wrong-size cleaned files = 0 and non-binary masks = 0.
- Exact duplicate leakage candidates are saved and reported.

In [2]:
from loco_project_utils import set_reproducible_seed
set_reproducible_seed(42)

In [3]:
from loco_project_utils import run_stage01_preprocessing

## Run Stage

Run this cell to generate the notebook outputs.

In [4]:
result = run_stage01_preprocessing(lab_root=LAB_ROOT, target_size=384)

D:\Dev Tools\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Outputs Saved

The cell below lists the concrete files created by this run.

In [5]:
from pathlib import Path
print('Outputs saved:')
for out in result.get('outputs', []):
    p = Path(out)
    print('-', p)
if 'checks' in result:
    print('\nAcceptance check values:')
    for key, value in result['checks'].items():
        print(f'{key}: {value}')

Outputs saved:
- D:\Anal Project Codex\Project_A_LOCO_AD\02_audit_reproducibility\environment_versions.csv
- D:\Anal Project Codex\Project_A_LOCO_AD\02_audit_reproducibility\requirements_freeze.txt
- D:\Anal Project Codex\Project_A_LOCO_AD\02_audit_reproducibility\dataset_audit_full.csv
- D:\Anal Project Codex\Project_A_LOCO_AD\02_audit_reproducibility\dataset_product_images_only.csv
- D:\Anal Project Codex\Project_A_LOCO_AD\02_audit_reproducibility\dataset_masks_only.csv
- D:\Anal Project Codex\Project_A_LOCO_AD\02_audit_reproducibility\product_image_count_summary.csv
- D:\Anal Project Codex\Project_A_LOCO_AD\02_audit_reproducibility\mask_count_summary.csv
- D:\Anal Project Codex\Project_A_LOCO_AD\02_audit_reproducibility\resize_letterbox_metadata.csv
- D:\Anal Project Codex\Project_A_LOCO_AD\02_audit_reproducibility\cleaned_letterbox_verify.csv
- D:\Anal Project Codex\Project_A_LOCO_AD\02_audit_reproducibility\image_hashes.csv
- D:\Anal Project Codex\Project_A_LOCO_AD\02_audit_reprod

## Notes

Training or fitting uses `train/good` only. `validation/good` is reserved for calibration, threshold selection, or hyperparameter selection. Test data is used for final evaluation only. Ground-truth masks are used only for localization evaluation or visualization.